In [1]:
import json
import torch
import torch.nn as nn
import torchmetrics
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from core_twice.data_module import DataModule
from core_twice.twice_da import twice_da_tiny
from core_twice.model_compilation import ModelCompilation
from core_twice.callbacks import LossMetricTracker
from calflops import calculate_flops
import os

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

with open('B:\\NetworkResearch\\core_twice\\configs\\twice_config.json') as f:
    cfg = json.load(f)['caltech']

dataset = cfg['dataset']
dataset_path = cfg['dataset_path']
image_size = cfg['image_size']
batch_size = cfg['batch_size']
num_classes = cfg['num_classes']
device = cfg['device']
task = cfg['task']
label_smoothing = cfg['label_smoothing']
learning_rate = cfg['learning_rate']
gradient_clipping = cfg['gradient_clipping']
accumulate_grad_batches = cfg['accumulate_grad_batches']
epochs = cfg['epochs']

data_module = DataModule(dataset=dataset,
                         dataset_path=dataset_path,
                         image_size=image_size,
                         batch_size=batch_size,
                         num_classes=num_classes)

network = twice_da_tiny(num_classes)
metrics = torchmetrics.MetricCollection({'accuracy': torchmetrics.Accuracy(task=task, num_classes=num_classes).to(device),
                                         'f1-score': torchmetrics.F1Score(task=task, num_classes=num_classes, average='macro').to(device)})
loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW

model = ModelCompilation(model=network,
                         metrics=metrics,
                         loss_function=loss_function,
                         optimizer=optimizer,
                         learning_rate=learning_rate,
                         accelerator=device,
                         data_module=data_module)

checkpoint_callback = ModelCheckpoint(filename='model-{epoch:02d}-{val_loss:.2f}-{val_accuracy:.2f}', monitor="val_loss")
loss_metric_tracker_callback = LossMetricTracker()

trainer = pl.Trainer(callbacks=[checkpoint_callback, loss_metric_tracker_callback],
                     precision='32',
                     accelerator=device,
                     devices="auto",
                     max_epochs=epochs,
                     gradient_clip_val=gradient_clipping,
                     accumulate_grad_batches=accumulate_grad_batches
                     )

B:\NetworkResearch\venv\lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of AlbumentationsX (2.2.2) is available! Your version is 2.0.9. Upgrade using: pip install -U albumentationsx
  check_for_updates()
B:\NetworkResearch\venv\lib\site-packages\albumentations\core\validation.py:132: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [2]:
ckpt_path = 'B:\\NetworkResearch\\lightning_logs\\version_1\\checkpoints\\model-epoch=314-val_loss=1.86-val_accuracy=0.75.ckpt'
trainer.test(model, datamodule=data_module, ckpt_path=ckpt_path)

You are using a CUDA device ('NVIDIA GeForce RTX 3060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
B:\NetworkResearch\core_twice\data_module.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they ar

caltech_indices loaded successfully


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at B:\NetworkResearch\lightning_logs\version_1\checkpoints\model-epoch=314-val_loss=1.86-val_accuracy=0.75.ckpt
B:\NetworkResearch\venv\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:419: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.7529411911964417
      test_f1-score         0.7145780920982361
        test_loss            1.849088191986084
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_accuracy': 0.7529411911964417,
  'test_f1-score': 0.7145780920982361,
  'test_loss': 1.849088191986084}]